<a href="https://colab.research.google.com/github/kkandanala9/Hello-World/blob/master/DTSC_4050/Week_6_Probability.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DTSC 4050 — Lesson 4: Probability and Sampling

This notebook covers the following topics:

1. Probability basics and core rules
2. Conditional probability and independence
3. Random/probability sampling
4. Simple and systematic random sampling


The examples are adapted from *Computational and Inferential Thinking* (primarily Chapters 9 and 10).




## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Random samples generator
rng = np.random.default_rng()

# Use this instead to set the seed and avoid random samples
# rng = np.random.default_rng(seed = 30)

pd.set_option("display.max_rows", 20)


# 2. Randomness in Python

Chapter 9 introduces randomness with `np.random.choice`. The result can change each time the code is run.



A population of 10 elements is created and four elements are selected without replacement using `random.sample()`.

In [5]:
import random

# Population: A list of elements (could be anything, numbers, strings, etc.)
population = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

# Sample size (number of elements to be selected)
sample_size = 4

# Perform simple random sampling
sample = random.sample(population, sample_size)

print("Sample:", sample)


Sample: [2, 6, 8, 7]


In [7]:
two_groups = np.array(["treatment", "control"])

# One random assignment
rng.choice(two_groups)


np.str_('control')

In [4]:
# Ten random assignments
rng.choice(two_groups, size=10)


array(['treatment', 'control', 'treatment', 'control', 'control',
       'control', 'control', 'treatment', 'control', 'treatment'],
      dtype='<U9')

# 3. Probability Basics

For an event $A$:

- $0 \le P(A) \le 1$
- $P(\text{not } A) = 1 - P(A)$
- If outcomes are equally likely:

$$
P(A)=\frac{\text{number of outcomes that make A happen}}
{\text{total number of outcomes}}
$$


### 3.1 Equally likely outcomes — fair die

In [9]:
die = np.arange(1, 7)
die


array([1, 2, 3, 4, 5, 6])

In [8]:
even_faces = die[die % 2 == 0]
p_even = len(even_faces) / len(die)

print("Even faces:", even_faces)
print("P(even) =", p_even)


Even faces: [2 4 6]
P(even) = 0.5


In [10]:
multiples_of_3 = die[die % 3 == 0]
p_multiple_3 = len(multiples_of_3) / len(die)

print("Multiples of 3:", multiples_of_3)
print("P(multiple of 3) =", p_multiple_3)


Multiples of 3: [3 6]
P(multiple of 3) = 0.3333333333333333


### 3.2 Complement rule

In [11]:
p_event = 0.70
p_not_event = 1 - p_event
p_not_event


0.30000000000000004

# 4. Multiplication Rule: When Both Events Must Happen

Let's consider the example of three tickets:

- Red
- Blue
- Green

Two tickets are drawn **without replacement**. What is the probability of drawing **Green first, then Red**?


In [12]:
tickets = np.array(["Red", "Blue", "Green"])

# All six ordered outcomes when two tickets are drawn without replacement
ordered_outcomes = [
    (first, second)
    for first in tickets
    for second in tickets
    if second != first
]

ordered_outcomes


[(np.str_('Red'), np.str_('Blue')),
 (np.str_('Red'), np.str_('Green')),
 (np.str_('Blue'), np.str_('Red')),
 (np.str_('Blue'), np.str_('Green')),
 (np.str_('Green'), np.str_('Red')),
 (np.str_('Green'), np.str_('Blue'))]

In [13]:
favorable = [outcome for outcome in ordered_outcomes
             if outcome == ("Green", "Red")]

print("Number of possible ordered outcomes:", len(ordered_outcomes))
print("Favorable outcomes:", favorable)
print("Probability =", len(favorable) / len(ordered_outcomes))


Number of possible ordered outcomes: 6
Favorable outcomes: [(np.str_('Green'), np.str_('Red'))]
Probability = 0.16666666666666666


The same answer follows from the multiplication rule:

$$
P(G\text{ then }R)
=
P(G)\times P(R\mid G)
=
\frac{1}{3}\times\frac{1}{2}
=
\frac{1}{6}
$$

We multiply because we need **Green AND THEN Red**. After Green is selected, Red is one of the two remaining tickets.


In [14]:
p_green_first = 1/3
p_red_second_given_green = 1/2

p_green_then_red = p_green_first * p_red_second_given_green
p_green_then_red


0.16666666666666666

### Simulate the ticket experiment

In [15]:
def draw_two_tickets():
    return tuple(rng.choice(tickets, size=2, replace=False))

for _ in range(5):
    print(draw_two_tickets())


(np.str_('Blue'), np.str_('Red'))
(np.str_('Red'), np.str_('Green'))
(np.str_('Green'), np.str_('Blue'))
(np.str_('Red'), np.str_('Green'))
(np.str_('Blue'), np.str_('Green'))


In [16]:
repetitions = 100
successes = 0

for _ in range(repetitions):
    if draw_two_tickets() == ("Green", "Red"):
        successes += 1

simulated_probability = successes / repetitions

print("Simulated probability:", simulated_probability)
print("Exact probability:", 1/6)


Simulated probability: 0.15
Exact probability: 0.16666666666666666


# 5. Addition Rule: When an Event Can Happen in Different Ways

Now ask: **What is the chance that one ticket is Green and the other is Red, in either order?**

The event can occur in two mutually exclusive ways:

- Green then Red (GR)
- Red then Green (RG)

Therefore:

$$
P(\text{one Green and one Red})
=
P(GR)+P(RG)
=
\frac16+\frac16
=
\frac13
$$


In [17]:
p_gr = 1/6
p_rg = 1/6
p_green_and_red = p_gr + p_rg

p_green_and_red


0.3333333333333333

### Simulate the unordered event

In [18]:
repetitions = 100_000
successes = 0

for _ in range(repetitions):
    draw = draw_two_tickets()
    if set(draw) == {"Green", "Red"}:
        successes += 1

print("Simulated probability:", successes / repetitions)
print("Exact probability:", 1/3)


Simulated probability: 0.3297
Exact probability: 0.3333333333333333


# 6. Discussion Example — Rick and Morty

A population contains 100 people, including Rick and Morty. We randomly select two people without replacement.

### (a) Probability both are selected

There are two possible orders:

1. Rick then Morty
2. Morty then Rick


In [19]:
p_both = (1/100)*(1/99) + (1/100)*(1/99)
p_both


0.00020202020202020205

### (b) Probability neither is selected

In [20]:
p_neither = (98/100) * (97/99)
p_neither


0.9602020202020202

# 7. Conditional Probability

Conditional probability asks for the probability of one event **given that another event has occurred**:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
$$

The ticket example is a natural illustration. Before any draw, the chance of Red is $1/3$.  
After Green has already been drawn, the chance of Red becomes $1/2$.


In [21]:
print("P(Red before drawing) =", 1/3)
print("P(Red second | Green first) =", 1/2)


P(Red before drawing) = 0.3333333333333333
P(Red second | Green first) = 0.5


# 8. Independent vs. Dependent Events

Two events are independent when knowing that one occurred does not change the probability of the other:

$$
P(A\mid B)=P(A)
$$

For independent events:

$$
P(A\cap B)=P(A)P(B)
$$

Drawing tickets **without replacement** is dependent because the first draw changes what remains.


### Independent example — two coin tosses

In [22]:
coin = np.array(["Heads", "Tails"])

p_first_heads = 1/2
p_second_heads = 1/2
p_both_heads = p_first_heads * p_second_heads

p_both_heads


0.25

### Two tickets without replacement

In [23]:
print("P(Red on first draw) =", 1/3)
print("P(Red on second draw | Green first) =", 1/2)
print("These probabilities differ, so the draws are dependent.")


P(Red on first draw) = 0.3333333333333333
P(Red on second draw | Green first) = 0.5
These probabilities differ, so the draws are dependent.


# 9. Sampling

The slides distinguish probability and non-probability sampling. Chapter 10 makes an important distinction:

- **Population:** all elements from which the sample is drawn.
- **Probability sample:** selection chances can be calculated before sampling.
- **Simple random sample (SRS):** random sampling **without replacement**.
- **Systematic sample:** randomly choose a starting position and then select regularly spaced elements.
- **Convenience sample:** selected because elements are easy to reach; it is not a probability sample.

The textbook's sampling examples use the `top_movies_2017.csv` data.


### 9.1 A small version of the textbook's top-movies table

In [24]:
top_movies = pd.DataFrame({
    "Title": [
        "Gone with the Wind", "Star Wars", "The Sound of Music",
        "E.T.: The Extra-Terrestrial", "Titanic", "The Ten Commandments",
        "Jaws", "Doctor Zhivago", "The Exorcist",
        "Snow White and the Seven Dwarves"
    ],
    "Studio": [
        "MGM", "Fox", "Fox", "Universal", "Paramount",
        "Paramount", "Universal", "MGM", "Warner Brothers", "Disney"
    ],
    "Gross": [
        198676459, 460998007, 158671368, 435110554, 658672302,
        65500000, 260000000, 111721910, 232906145, 184925486
    ],
    "Gross (Adjusted)": [
        1796176700, 1583483200, 1266072700, 1261085000, 1204368000,
        1164590000, 1138620700, 1103564200, 983226600, 969010000
    ],
    "Year": [1939, 1977, 1965, 1982, 1997, 1956, 1975, 1965, 1973, 1937]
})

top_movies.insert(0, "Row Index", np.arange(len(top_movies)))
top_movies


,Row Index,Title,Studio,Gross,Gross (Adjusted),Year
0,0,Gone with the Wind,MGM,198676459,1796176700,1939
1,1,Star Wars,Fox,460998007,1583483200,1977
2,2,The Sound of Music,Fox,158671368,1266072700,1965
3,3,E.T.: The Extra-Terrestrial,Universal,435110554,1261085000,1982
4,4,Titanic,Paramount,658672302,1204368000,1997
5,5,The Ten Commandments,Paramount,65500000,1164590000,1956
6,6,Jaws,Universal,260000000,1138620700,1975
7,7,Doctor Zhivago,MGM,111721910,1103564200,1965
8,8,The Exorcist,Warner Brothers,232906145,983226600,1973
9,9,Snow White and the Seven Dwarves,Disney,184925486,969010000,1937


### 9.2 Deterministic sample

Selecting known rows directly does **not** involve chance.


In [25]:
top_movies.iloc[[3, 6, 9]]


,Row Index,Title,Studio,Gross,Gross (Adjusted),Year
3,3,E.T.: The Extra-Terrestrial,Universal,435110554,1261085000,1982
6,6,Jaws,Universal,260000000,1138620700,1975
9,9,Snow White and the Seven Dwarves,Disney,184925486,969010000,1937


### 9.3 Simple random sample — without replacement

Each selected row is removed from the pool before the next selection.


In [26]:
sample_indices = rng.choice(top_movies.index, size=4, replace=False)
top_movies.loc[sample_indices]


,Row Index,Title,Studio,Gross,Gross (Adjusted),Year
2,2,The Sound of Music,Fox,158671368,1266072700,1965
4,4,Titanic,Paramount,658672302,1204368000,1997
6,6,Jaws,Universal,260000000,1138620700,1975
9,9,Snow White and the Seven Dwarves,Disney,184925486,969010000,1937


### 9.4 Sampling with replacement

With replacement, the same movie can appear more than once.


In [27]:
sample_indices = rng.choice(top_movies.index, size=8, replace=True)
top_movies.loc[sample_indices]


,Row Index,Title,Studio,Gross,Gross (Adjusted),Year
3,3,E.T.: The Extra-Terrestrial,Universal,435110554,1261085000,1982
1,1,Star Wars,Fox,460998007,1583483200,1977
3,3,E.T.: The Extra-Terrestrial,Universal,435110554,1261085000,1982
8,8,The Exorcist,Warner Brothers,232906145,983226600,1973
7,7,Doctor Zhivago,MGM,111721910,1103564200,1965
0,0,Gone with the Wind,MGM,198676459,1796176700,1939
3,3,E.T.: The Extra-Terrestrial,Universal,435110554,1261085000,1982
2,2,The Sound of Music,Fox,158671368,1266072700,1965


### 9.5 Systematic random sample

The textbook chooses a random starting position and then every 10th row.  
Our small table has only 10 rows, so the following larger row-index population makes the pattern easier to see.


In [28]:
population = pd.DataFrame({
    "Row Index": np.arange(100),
    "Value": np.arange(100)
})

start = rng.choice(np.arange(10))
systematic_sample = population.iloc[np.arange(start, len(population), 10)]

print("Random start:", start)
systematic_sample


Random start: 4


,Row Index,Value
4,4,4
14,14,14
24,24,24
34,34,34
44,44,44
54,54,54
64,64,64
74,74,74
84,84,84
94,94,94
